# Design a Raw Landing Zone — Solution


## S3 Prefix Structure

```
s3://cloudserve-lakehouse/bronze/
├── clickstream/
│   └── year=YYYY/month=MM/day=DD/hour=HH/
│       └── batch_{timestamp}.parquet
├── billing/
│   └── year=YYYY/month=MM/day=DD/
│       └── billing_export_{date}.parquet
├── support_tickets/
│   └── year=YYYY/month=MM/day=DD/hour=HH/
│       └── tickets_{timestamp}.parquet
└── product_usage/
    └── year=YYYY/month=MM/day=DD/hour=HH/
        └── usage_{timestamp}.parquet
```

## Source-to-Landing Mapping

| Source | Ingestion Tool | Landing Format | Partition | Buffer/Batch | Rationale |
|--------|---------------|----------------|-----------|--------------|-----------|
| Clickstream (JSON, real-time) | Kinesis Data Firehose | Parquet (converted on delivery) | year/month/day/hour | 128 MB or 5 min (whichever first) | Firehose handles JSON→Parquet conversion natively; buffering prevents small files at 2K events/sec |
| Billing (CSV, daily) | AWS Glue ETL (scheduled) | Parquet | year/month/day | Single file per daily run | Low volume (500MB); Glue converts CSV→Parquet with schema enforcement; daily batch is sufficient |
| Support tickets (API, hourly) | Step Functions + Lambda | Parquet (converted after extraction) | year/month/day/hour | Full hourly batch (all pages) as single file | Step Functions handles pagination + OAuth; Lambda converts JSON→Parquet; hourly cadence matches API poll |
| Product usage (Parquet, 15-min) | S3-to-S3 copy (EventBridge + Lambda) | Parquet (keep as-is) | year/month/day/hour | Accumulate 4 files (1 hour) then compact | Already Parquet; compact every hour to avoid 96 files/day small-file problem |

## Lifecycle Policies

| Layer | Age | Action |
|-------|-----|--------|
| Bronze | 0–90 days | S3 Standard |
| Bronze | 90–365 days | Transition to S3 Infrequent Access |
| Bronze | >365 days | Delete (re-derivable from source systems within retention window) |

Additional rules:
- Enable versioning on the bronze bucket (protects against accidental overwrites)
- Enable S3 Intelligent-Tiering for the clickstream prefix (access patterns vary)
- All objects tagged with `source`, `ingestion_timestamp`, and `schema_version` metadata